Purpose: The main goal of this table is to provide a detailed statistical comparison of the patient characteristics between your development cohort (NHANES) and your external validation cohort (BRFSS).


Importance: This is arguably the most important table for setting up your paper's narrative. It quantitatively proves the existence of "dataset shift" —the subtle but significant differences between the two populations. This table provides the concrete evidence that explains why your model's performance dropped during external validation.



Content: The table will have a row for each key predictor. The columns will show the summary statistics for both datasets (Mean ± SD for continuous variables, n (%) for categorical variables), and critically, include statistical tests (p-values) and effect sizes (like Cohen's d) to measure the magnitude of the differences.

In [2]:
import pandas as pd
import numpy as np
from scipy.stats import ttest_ind, chi2_contingency
import pingouin as pg

print("Libraries imported successfully.")

# --- Load Datasets ---
# PASTE THE FULL, ABSOLUTE PATHS YOU COPIED IN THE LINES BELOW
# Make sure to put an 'r' before the quotes
nhanes_filepath = r'C:\diabetes_prediction_project\data\03_processed\nhanes_final.csv'
brfss_filepath = r'C:\diabetes_prediction_project\data\03_processed\brfss_final.csv'

# Column names for the 8 common predictors
columns_to_load = [
    'Age', 'Gender', 'Race_Ethnicity', 'BMI', 'Smoking_Status', 
    'Physical_Activity', 'History_Heart_Attack', 'History_Stroke', 'Diabetes_Outcome'
]

try:
    df_nhanes = pd.read_csv(nhanes_filepath, usecols=columns_to_load)
    df_brfss = pd.read_csv(brfss_filepath, usecols=columns_to_load)
    
    print("Successfully loaded datasets!")
    print(f"NHANES shape: {df_nhanes.shape}")
    print(f"BRFSS shape: {df_brfss.shape}")

except Exception as e:
    print(f"An error occurred during data loading: {e}")

Libraries imported successfully.
Successfully loaded datasets!
NHANES shape: (15685, 9)
BRFSS shape: (1285783, 9)


In [4]:
import pandas as pd
import numpy as np
from scipy.stats import ttest_ind, chi2_contingency
import pingouin as pg

# --- Load Datasets using the absolute paths ---
nhanes_filepath = r'C:\diabetes_prediction_project\data\03_processed\nhanes_final.csv'
brfss_filepath = r'C:\diabetes_prediction_project\data\03_processed\brfss_final.csv'

df_nhanes = pd.read_csv(nhanes_filepath)
df_brfss = pd.read_csv(brfss_filepath)
print("Successfully loaded datasets.")

# --- Table Generation ---
table_rows = []

# --- 1. Age (Continuous) ---
mean_nhanes = df_nhanes['Age'].mean()
std_nhanes = df_nhanes['Age'].std()
mean_brfss = df_brfss['Age'].mean()
std_brfss = df_brfss['Age'].std()
stat, p_val = ttest_ind(df_nhanes['Age'].dropna(), df_brfss['Age'].dropna())
cohens_d = pg.compute_effsize(df_nhanes['Age'].dropna(), df_brfss['Age'].dropna(), eftype='cohen')
table_rows.append({
    'Characteristic': 'Age (years)',
    'NHANES': f'{mean_nhanes:.1f} ± {std_nhanes:.1f}',
    'BRFSS': f'{mean_brfss:.1f} ± {std_brfss:.1f}',
    'p-value': '<0.001' if p_val < 0.001 else f'{p_val:.3f}',
    'Effect Size': f"Cohen's d={cohens_d:.2f}"
})

# --- 2. BMI (Continuous) ---
mean_nhanes = df_nhanes['BMI'].mean()
std_nhanes = df_nhanes['BMI'].std()
mean_brfss = df_brfss['BMI'].mean()
std_brfss = df_brfss['BMI'].std()
stat, p_val = ttest_ind(df_nhanes['BMI'].dropna(), df_brfss['BMI'].dropna())
cohens_d = pg.compute_effsize(df_nhanes['BMI'].dropna(), df_brfss['BMI'].dropna(), eftype='cohen')
table_rows.append({
    'Characteristic': 'BMI (kg/m²)',
    'NHANES': f'{mean_nhanes:.1f} ± {std_nhanes:.1f}',
    'BRFSS': f'{mean_brfss:.1f} ± {std_brfss:.1f}',
    'p-value': '<0.001' if p_val < 0.001 else f'{p_val:.3f}',
    'Effect Size': f"Cohen's d={cohens_d:.2f}"
})

# This is a helper function for all categorical variables
def process_categorical(var_name, positive_label, positive_code, negative_code):
    nhanes_counts = df_nhanes[var_name].value_counts()
    brfss_counts = df_brfss[var_name].value_counts()

    nhanes_pos_n = nhanes_counts.get(positive_code, 0)
    nhanes_pos_pct = (nhanes_pos_n / len(df_nhanes.dropna(subset=[var_name]))) * 100
    brfss_pos_n = brfss_counts.get(positive_code, 0)
    brfss_pos_pct = (brfss_pos_n / len(df_brfss.dropna(subset=[var_name]))) * 100

    nhanes_summary_str = f'{nhanes_pos_n:,} ({nhanes_pos_pct:.1f}%)'
    brfss_summary_str = f'{brfss_pos_n:,} ({brfss_pos_pct:.1f}%)'

    contingency = pd.DataFrame({'NHANES': nhanes_counts, 'BRFSS': brfss_counts}).fillna(0)
    chi2, p_val, _, _ = chi2_contingency(contingency)
    
    nhanes_neg_n = nhanes_counts.get(negative_code, 0)
    brfss_neg_n = brfss_counts.get(negative_code, 0)
    odds_ratio = ((brfss_pos_n + 0.5) / (brfss_neg_n + 0.5)) / ((nhanes_pos_n + 0.5) / (nhanes_neg_n + 0.5))

    table_rows.append({
        'Characteristic': positive_label,
        'NHANES': nhanes_summary_str,
        'BRFSS': brfss_summary_str,
        'p-value': '<0.001' if p_val < 0.001 else f'{p_val:.3f}',
        'Effect Size': f'OR={odds_ratio:.2f}'
    })

# --- Process All Categorical Variables using the helper function ---
# If your final harmonized data uses different codes, just change them here.
process_categorical('Gender', 'Female', positive_code=0, negative_code=1) # Assuming Female=0, Male=1
process_categorical('Smoking_Status', 'Current Smoker', positive_code=1, negative_code=0) # Assuming Smoker=1
process_categorical('Physical_Activity', 'Physically Active', positive_code=1, negative_code=0) # Assuming Active=1
process_categorical('History_Heart_Attack', 'History of Heart Attack', positive_code=1, negative_code=0) # Assuming Yes=1
process_categorical('History_Stroke', 'History of Stroke', positive_code=1, negative_code=0) # Assuming Yes=1
process_categorical('Diabetes_Outcome', 'Diabetes Prevalence', positive_code=1, negative_code=0) # Assuming Yes=1

# --- Race/Ethnicity (special handling due to multiple categories) ---
nhanes_race = df_nhanes['Race_Ethnicity'].value_counts().sort_index()
brfss_race = df_brfss['Race_Ethnicity'].value_counts().sort_index()
contingency_race = pd.DataFrame({'NHANES': nhanes_race, 'BRFSS': brfss_race}).fillna(0)
chi2_race, p_val_race, _, _ = chi2_contingency(contingency_race)

# You would add rows for each race category here if desired, but for now we'll just note it.
print("Processed all variables. Race/Ethnicity requires special handling if each category is needed in the table.")

# --- Create, Display, and Save the Final Table ---
final_table = pd.DataFrame(table_rows).set_index('Characteristic')
final_table.columns = ['NHANES (n=15,685)', 'BRFSS (n=1,282,897)', 'p-value', 'Effect Size']

print("\n--- Final Table 1: Baseline Characteristics ---")
display(final_table)

output_path = 'results/TABLE_1_cohort_characteristics_final.csv'
final_table.to_csv(output_path)
print(f"\nTable 1 has been successfully saved to: {output_path}")

Successfully loaded datasets.
Processed all variables. Race/Ethnicity requires special handling if each category is needed in the table.

--- Final Table 1: Baseline Characteristics ---


,"NHANES (n=15,685)","BRFSS (n=1,282,897)",p-value,Effect Size
Characteristic,,,,
Age (years),49.0 ± 18.6,55.1 ± 17.7,<0.001,Cohen's d=-0.35
BMI (kg/m²),29.7 ± 7.4,28.5 ± 6.5,<0.001,Cohen's d=0.19
Female,0 (0.0%),"596,991 (46.4%)",<0.001,OR=13183.71
Current Smoker,"6,311 (40.2%)","487,117 (40.4%)",<0.001,OR=0.00
Physically Active,"3,981 (25.4%)","974,180 (75.9%)",<0.001,OR=0.00
History of Heart Attack,667 (4.5%),"69,896 (5.5%)",<0.001,OR=0.00
History of Stroke,696 (4.7%),"52,129 (4.1%)",<0.001,OR=0.00
Diabetes Prevalence,"2,908 (18.5%)","170,868 (13.3%)",<0.001,OR=0.68



Table 1 has been successfully saved to: results/TABLE_1_cohort_characteristics_final.csv


In [8]:
import pandas as pd

# Path to your results file
results_filepath = 'C:\diabetes_prediction_project\results\TABLE_2_internal_validation.csv'

try:
    # Load the CSV file
    df_internal_val = pd.read_csv(results_filepath)
    
    print("Successfully loaded the internal validation results file.")
    print("Here are the first few rows:")
    display(df_internal_val.head())
    
    print("\nColumns in this file are:")
    print(df_internal_val.columns.tolist())

except FileNotFoundError:
    print(f"Error: The file '{results_filepath}' was not found.")
    print("Please check if the file exists in your 'results' folder.")
except Exception as e:
    print(f"An error occurred: {e}")

An error occurred: [Errno 22] Invalid argument: 'C:\\diabetes_prediction_project\results\\TABLE_2_internal_validation.csv'


<>:4: SyntaxWarning: invalid escape sequence '\d'
<>:4: SyntaxWarning: invalid escape sequence '\d'
C:\Users\Asus\AppData\Local\Temp\ipykernel_34756\3063629625.py:4: SyntaxWarning: invalid escape sequence '\d'
  results_filepath = 'C:\diabetes_prediction_project\results\TABLE_2_internal_validation.csv'


In [9]:
import pandas as pd
import io

# The data you provided from your CSV file
csv_data = """Model,Mean AUC,95% CI,Sensitivity*,Specificity*,PPV*,NPV*
XGBoost,0.795,"[0.788, 0.807]",0.805,0.657,0.352,0.938
Logistic Regression,0.792,"[0.783, 0.803]",0.802,0.658,0.349,0.936
Random Forest,0.789,"[0.783, 0.798]",0.818,0.642,0.342,0.94
SVM,0.706,"[0.691, 0.734]",0.546,0.782,0.365,0.884
"""

# Load the data into a pandas DataFrame
df_internal_val = pd.read_csv(io.StringIO(csv_data))

# Rename columns for clarity (Precision is PPV, Recall is Sensitivity)
df_internal_val = df_internal_val.rename(columns={
    'Sensitivity*': 'Recall',
    'PPV*': 'Precision',
    'Specificity*': 'Specificity',
    'NPV*': 'NPV'
})

# Calculate the F1 Score
precision = df_internal_val['Precision']
recall = df_internal_val['Recall']
df_internal_val['F1 Score'] = 2 * (precision * recall) / (precision + recall)

# Reorder columns to match the enhanced table format
enhanced_df = df_internal_val[[
    'Model', 'Mean AUC', '95% CI', 'Recall', 'Specificity', 'Precision', 
    'NPV', 'F1 Score'
]]

print("--- Partially Enhanced Table 2 ---")
display(enhanced_df)

--- Partially Enhanced Table 2 ---


,Model,Mean AUC,95% CI,Recall,Specificity,Precision,NPV,F1 Score
0,XGBoost,0.795,"[0.788, 0.807]",0.805,0.657,0.352,0.938,0.489818
1,Logistic Regression,0.792,"[0.783, 0.803]",0.802,0.658,0.349,0.936,0.486356
2,Random Forest,0.789,"[0.783, 0.798]",0.818,0.642,0.342,0.940,0.482338
3,SVM,0.706,"[0.691, 0.734]",0.546,0.782,0.365,0.884,0.437519


In [10]:
import pandas as pd
import numpy as np

# --- 1. Load Datasets ---
# Using the absolute paths for reliability
nhanes_filepath = r'C:\diabetes_prediction_project\data\03_processed\nhanes_final.csv'
brfss_filepath = r'C:\diabetes_prediction_project\data\03_processed\brfss_final.csv'

df_nhanes = pd.read_csv(nhanes_filepath)
df_brfss = pd.read_csv(brfss_filepath)
print("Successfully loaded datasets.")

# --- 2. Define Features and Imputation Methods ---
# The 8 predictors used in the model
features = [
    'Age', 'Gender', 'Race_Ethnicity', 'BMI', 'Smoking_Status', 
    'Physical_Activity', 'History_Heart_Attack', 'History_Stroke'
]

# Imputation strategy described in your manuscript 
imputation_methods = {
    'Age': 'Median',
    'BMI': 'Median',
    'Gender': 'Mode',
    'Race_Ethnicity': 'Mode',
    'Smoking_Status': 'Mode',
    'Physical_Activity': 'Mode',
    'History_Heart_Attack': 'Mode',
    'History_Stroke': 'Mode'
}

# --- 3. Calculate Missing Data Statistics ---
missing_data_rows = []
for feature in features:
    missing_nhanes_pct = df_nhanes[feature].isnull().sum() / len(df_nhanes) * 100
    missing_brfss_pct = df_brfss[feature].isnull().sum() / len(df_brfss) * 100
    
    missing_data_rows.append({
        'Feature': feature,
        'Missing in NHANES (%)': f'{missing_nhanes_pct:.1f}%',
        'Missing in BRFSS (%)': f'{missing_brfss_pct:.1f}%',
        'Imputation Method': imputation_methods.get(feature, 'N/A')
    })

# --- 4. Create, Display, and Save the Table ---
table3_missing_data = pd.DataFrame(missing_data_rows).set_index('Feature')

print("\n\n--- Table 3: Missing Data Analysis and Imputation Strategy ---")
display(table3_missing_data)

output_path = 'results/TABLE_3_missing_data_analysis.csv'
table3_missing_data.to_csv(output_path)
print(f"\nTable 3 has been successfully saved to: {output_path}")

Successfully loaded datasets.


--- Table 3: Missing Data Analysis and Imputation Strategy ---


,Missing in NHANES (%),Missing in BRFSS (%),Imputation Method
Feature,,,
Age,0.0%,2.1%,Median
Gender,0.0%,0.0%,Mode
Race_Ethnicity,0.0%,34.6%,Mode
BMI,7.9%,10.7%,Median
Smoking_Status,0.0%,6.3%,Mode
Physical_Activity,0.0%,0.2%,Mode
History_Heart_Attack,4.7%,0.6%,Mode
History_Stroke,4.7%,0.3%,Mode



Table 3 has been successfully saved to: results/TABLE_3_missing_data_analysis.csv


In [15]:
import pandas as pd
import numpy as np

# --- 1. Load the Raw Grid Search Details ---
print("Loading the raw XGBoost grid search results file...")
try:
    # Use the absolute path to the file we just created
    gs_filepath = r'C:\diabetes_prediction_project\notebooks\results\xgboost_grid_search_details.csv'
    df_gs_raw = pd.read_csv(gs_filepath)
    print("Successfully loaded the raw grid search results.")

    # --- 2. Clean the Data ---
    # The file has repeated headers from being appended. Let's remove them.
    # We do this by checking a column that should be numeric. If it contains text, it's a header row.
    df_cleaned = df_gs_raw[pd.to_numeric(df_gs_raw['mean_test_score'], errors='coerce').notna()].copy()

    # Convert columns to their proper numeric types
    df_cleaned['mean_test_score'] = pd.to_numeric(df_cleaned['mean_test_score'])
    df_cleaned['param_learning_rate'] = pd.to_numeric(df_cleaned['param_learning_rate'])
    df_cleaned['param_max_depth'] = pd.to_numeric(df_cleaned['param_max_depth'])
    df_cleaned['param_n_estimators'] = pd.to_numeric(df_cleaned['param_n_estimators'])

    # --- 3. Process and Aggregate the Results ---
    # Group by the parameter combinations and average their scores across the 5 outer folds.
    hyperparameter_summary = df_cleaned.groupby(
        ['param_learning_rate', 'param_max_depth', 'param_n_estimators']
    ).agg(
        Mean_AUC=('mean_test_score', 'mean'),
        Std_AUC=('mean_test_score', 'std')
    ).reset_index()

    # --- 4. Format the Table for the Manuscript ---
    hyperparameter_summary.rename(columns={
        'param_learning_rate': 'learning_rate',
        'param_max_depth': 'max_depth',
        'param_n_estimators': 'n_estimators'
    }, inplace=True)

    table4_hyperparameters = hyperparameter_summary.sort_values(by='Mean_AUC', ascending=False)
    table4_hyperparameters.insert(0, 'Rank', range(1, 1 + len(table4_hyperparameters)))
    table4_hyperparameters.set_index('Rank', inplace=True)

    # --- 5. Display and Save the Final Table ---
    print("\n\n--- Final, Publication-Quality Table 4: Hyperparameter Tuning ---")
    display(table4_hyperparameters)

    output_path = 'results/TABLE_4_hyperparameter_tuning.csv'
    table4_hyperparameters.to_csv(output_path)
    print(f"\nEnhanced Table 4 has been successfully saved to: {output_path}")

except Exception as e:
    print(f"An error occurred: {e}")

Loading the raw XGBoost grid search results file...
Successfully loaded the raw grid search results.


--- Final, Publication-Quality Table 4: Hyperparameter Tuning ---


,learning_rate,max_depth,n_estimators,Mean_AUC,Std_AUC
Rank,,,,,
1,0.10,3,100,0.793930,0.001127
2,0.05,3,200,0.793909,0.001010
3,0.05,3,100,0.792277,0.001023
4,0.10,3,200,0.792112,0.001457
5,0.05,5,100,0.790258,0.001142
6,0.05,5,200,0.788654,0.001631
7,0.10,5,100,0.787879,0.001402
8,0.10,5,200,0.781111,0.001973



Enhanced Table 4 has been successfully saved to: results/TABLE_4_hyperparameter_tuning.csv


In [2]:
import pandas as pd
import numpy as np

# --- 1. Define Internal Validation Results (from your Manuscript for XGBoost) ---
internal_results = {
    'AUC': 0.795,
    'Accuracy': 0.688, 
    'Sensitivity': 0.805,
    'Specificity': 0.657,
    'PPV': 0.352,
    'NPV': 0.938,
    'F1 Score': 0.485,
    'Brier Score': np.nan 
}

# --- 2. Define External Validation Results (from the script we just ran) ---
external_results = {
    'AUC': 0.717159, # Using the value our script produced
    'Brier Score': 0.123932,
    'Accuracy': 0.541724,
    'Sensitivity': 0.798183,
    'Specificity': 0.502318,
    'PPV': 0.197709,
    'NPV': 0.941856,
    'F1 Score': 0.316918
}

# --- 3. Build the Comparison Table ---
table_rows = []
metrics_to_compare = [
    'AUC', 'Accuracy', 'Sensitivity', 'Specificity', 'PPV', 'NPV', 'F1 Score', 'Brier Score'
]

for metric in metrics_to_compare:
    internal_val = internal_results.get(metric)
    external_val = external_results.get(metric)
    difference = external_val - internal_val if pd.notnull(internal_val) and pd.notnull(external_val) else np.nan
    
    table_rows.append({
        'Metric': metric,
        'Internal (NHANES)': f'{internal_val:.3f}' if pd.notnull(internal_val) else '[NOT REPORTED]',
        'External (BRFSS)': f'{external_val:.3f}' if pd.notnull(external_val) else '[NOT REPORTED]',
        'Difference (Δ)': f'{difference:+.3f}' if pd.notnull(difference) else 'N/A'
    })

# --- 4. Create, Display, and Save the Final Table ---
table5_comparison = pd.DataFrame(table_rows).set_index('Metric')

print("\n\n--- Final, Manuscript-Consistent Table 5: External Validation Comparison ---")
display(table5_comparison)

output_path = 'results/TABLE_5_external_validation_comparison.csv'
table5_comparison.to_csv(output_path)
print(f"\nEnhanced Table 5 has been successfully saved to: {output_path}")



--- Final, Manuscript-Consistent Table 5: External Validation Comparison ---


,Internal (NHANES),External (BRFSS),Difference (Δ)
Metric,,,
AUC,0.795,0.717,-0.078
Accuracy,0.688,0.542,-0.146
Sensitivity,0.805,0.798,-0.007
Specificity,0.657,0.502,-0.155
PPV,0.352,0.198,-0.154
NPV,0.938,0.942,+0.004
F1 Score,0.485,0.317,-0.168
Brier Score,[NOT REPORTED],0.124,N/A



Enhanced Table 5 has been successfully saved to: results/TABLE_5_external_validation_comparison.csv


In [4]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer

# Import the four models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
import xgboost as xgb

print("Starting Feature Importance Analysis...")

# --- 1. Load and Prepare NHANES Data ---
nhanes_filepath = r'C:\diabetes_prediction_project\data\03_processed\nhanes_final.csv'
df_nhanes = pd.read_csv(nhanes_filepath)
df_nhanes.dropna(subset=['Diabetes_Outcome'], inplace=True)
print("Successfully loaded NHANES dataset.")

common_features = [
    'Age', 'Gender', 'Race_Ethnicity', 'BMI', 'Smoking_Status',
    'Physical_Activity', 'History_Heart_Attack', 'History_Stroke'
]
X = df_nhanes[common_features]
y = df_nhanes['Diabetes_Outcome']

# Impute missing values
imputer = SimpleImputer(strategy='median')
X_imputed = imputer.fit_transform(X)
X_imputed = pd.DataFrame(X_imputed, columns=common_features)
print("Data imputation complete.")


# --- 2. Define Final Models with Optimal Hyperparameters ---
# These parameters are based on your manuscript and the cross-validation output.
models = {
    'XGBoost': xgb.XGBClassifier(
        learning_rate=0.05, max_depth=3, n_estimators=200,
        use_label_encoder=False, eval_metric='logloss', random_state=42
    ),
    'Logistic Regression': LogisticRegression(
        C=10.0, penalty='l2', solver='liblinear', class_weight='balanced', max_iter=1000
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, max_depth=10, min_samples_leaf=10,
        class_weight='balanced', random_state=42
    ),
    # For SVM, we use a linear kernel to get direct coefficients as importance scores.
    'SVM': SVC(kernel='linear', C=0.1, probability=True, class_weight='balanced', random_state=42)
}

# --- 3. Train Models and Extract Feature Importances ---
importances = {}
for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_imputed, y)
    
    if name in ['Logistic Regression', 'SVM']:
        # Importance is the absolute value of the learned coefficients
        importance_scores = np.abs(model.coef_[0])
    else: # For Random Forest and XGBoost
        importance_scores = model.feature_importances_
        
    # Normalize importances to sum to 1 for easy comparison
    importances[name] = importance_scores / np.sum(importance_scores)

print("All models trained and importances extracted.")

# --- 4. Create, Display, and Save the Final Table ---
table6_importances = pd.DataFrame(importances, index=common_features)

# Add a 'Consensus Rank' column by averaging the ranks from each model
ranks = table6_importances.rank(axis=0, ascending=False)
table6_importances['Consensus_Rank'] = ranks.mean(axis=1).rank(method='first')

# Sort by the consensus rank
table6_importances = table6_importances.sort_values(by='Consensus_Rank')

print("\n\n--- Final, Publication-Quality Table 6: Feature Importance Comparison ---")
display(table6_importances.style.format('{:.3f}'))

output_path = 'results/TABLE_6_feature_importance_comparison.csv'
table6_importances.to_csv(output_path)
print(f"\nEnhanced Table 6 has been successfully saved to: {output_path}")

Starting Feature Importance Analysis...
Successfully loaded NHANES dataset.
Data imputation complete.
Training XGBoost...
Training Logistic Regression...
Training Random Forest...


c:\diabetes_prediction_project\.venv\Lib\site-packages\xgboost\training.py:183: UserWarning: [12:24:11] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Training SVM...
All models trained and importances extracted.


--- Final, Publication-Quality Table 6: Feature Importance Comparison ---


,XGBoost,Logistic Regression,Random Forest,SVM,Consensus_Rank
Physical_Activity,0.139,0.426,0.089,0.457,1.000
Age,0.430,0.044,0.552,0.048,2.000
BMI,0.140,0.067,0.231,0.063,3.000
Gender,0.073,0.306,0.019,0.296,4.000
History_Stroke,0.053,0.117,0.020,0.092,5.000
Race_Ethnicity,0.077,0.007,0.051,0.007,6.000
History_Heart_Attack,0.064,0.013,0.025,0.010,7.000
Smoking_Status,0.024,0.020,0.013,0.028,8.000



Enhanced Table 6 has been successfully saved to: results/TABLE_6_feature_importance_comparison.csv
